In [1]:
import torch
import torch.nn as nn
import math
import psutil
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
import math
import json
import torch
from torch.utils.data import Dataset

class TAMECriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols, dict_path):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        self.supported_dims = ["64", "32", "16", "8", "4", "2"]
        
        print("Đang tải TAME Dictionary vào bộ nhớ...")
        with open(dict_path, "r") as f:
            self.tame_dict = json.load(f)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        # 1. Xử lý Dense
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                dense_x.append(math.log1p(float(val))) 
        
        # 2. Xử lý Categorical (Positional Routing - Xếp hình Tetris)
        num_cats = len(self.cat_cols) # Chính là 26
        # Tạo sẵn 6 mâm, mỗi mâm có đúng 26 slot số 0 (Vector rỗng)
        grouped_cat = {dim: [0] * num_cats for dim in self.supported_dims}
        
        for col_idx, col in enumerate(self.cat_cols):
            val = row[col]
            if val is not None:
                gid = f"{col}_{val}"
                if gid in self.tame_dict:
                    info = self.tame_dict[gid]
                    dim_group = str(info["Dim_Group"])
                    local_idx = info["Local_Index"]
                    
                    # Cắm ID vào ĐÚNG vị trí cột gốc của nó
                    grouped_cat[dim_group][col_idx] = local_idx
        
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            {dim: torch.tensor(grouped_cat[dim], dtype=torch.long) for dim in self.supported_dims},
            torch.tensor(label, dtype=torch.float32)
        )

In [3]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
import os

dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]


train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"
val_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/validation.parquet"

print("Đang nạp dữ liệu từ các file Parquet...")

train_hf = load_dataset("parquet", data_files=train_file, split="train")
val_hf = load_dataset("parquet", data_files=val_file, split="train")

print(f"Train: {len(train_hf)} | Val: {len(val_hf)}")

print("Đang khởi tạo Dataset và DataLoader...")

train_dataset = TAMECriteoDataset(train_hf, dense_cols, cat_cols, "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_ivs_dictionary.json")
val_dataset = TAMECriteoDataset(val_hf, dense_cols, cat_cols, "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_ivs_dictionary.json")

train_loader = DataLoader(
    train_dataset, 
    batch_size=8192, 
    shuffle=True, 
    num_workers=8,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=8192, 
    shuffle=False, 
    num_workers=8, 
    persistent_workers=True, 
    pin_memory=True
)
print("Hoàn tất DataLoader!")

Đang nạp dữ liệu từ các file Parquet...


Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/32 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 36665662 | Val: 4583562
Đang khởi tạo Dataset và DataLoader...
Đang tải TAME Dictionary vào bộ nhớ...
Đang tải TAME Dictionary vào bộ nhớ...
Hoàn tất DataLoader!


In [4]:
import torch
import torch.nn as nn

class TAME_Embedding(nn.Module):
    """
    TAME: Target-Aware Memory-efficient Embedding Layer.
    Sử dụng Positional Sum-Routing để chập 6 không gian chiều lại làm 1,
    loại bỏ hoàn toàn sự lạm phát tham số.
    """
    def __init__(self, vocab_sizes, target_dim=64):
        super(TAME_Embedding, self).__init__()
        self.target_dim = target_dim
        self.supported_dims = [64, 32, 16, 8, 4, 2]
        self.embedding_blocks = nn.ModuleDict()
        
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in vocab_sizes and vocab_sizes[dim_str] > 0:
                self.embedding_blocks[dim_str] = nn.Embedding(
                    num_embeddings=vocab_sizes[dim_str] + 1, 
                    embedding_dim=dim,
                    padding_idx=0
                )

    def forward(self, grouped_inputs):
        final_output = 0 # Khởi tạo biến để cộng dồn (Sum)
        
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in grouped_inputs and dim_str in self.embedding_blocks:
                x = grouped_inputs[dim_str]
                emb = self.embedding_blocks[dim_str](x)
                
                # Zero-Flop Tiling
                if dim == self.target_dim:
                    out = emb
                else:
                    n_repeats = self.target_dim // dim
                    out = emb.repeat(1, 1, n_repeats) / n_repeats
                    
                # SỬA Ở ĐÂY: Cộng chập ma trận (Summation) thay vì Concat
                final_output = final_output + out
        
        # Output trả về chắc chắn là [Batch, 26, 64]
        return final_output

In [5]:
import torch
import torch.nn as nn

class TAME_DLRM(nn.Module):
    def __init__(self, vocab_sizes, num_dense_features, embed_dim=64, bottom_mlp_dims=[256, 128, 64], top_mlp_dims=[512, 256, 128]):
        super(TAME_DLRM, self).__init__()
        
  
        self.tame_emb = TAME_Embedding(vocab_sizes, target_dim=embed_dim)
        self.num_cat_features = 26 # Cố định số cột Categorical
        
        # bottom mlp
        bottom_layers = []
        in_dim = num_dense_features
        if in_dim > 0:
            for dim in bottom_mlp_dims:
                bottom_layers.append(nn.Linear(in_dim, dim))
                bottom_layers.append(nn.BatchNorm1d(dim)) # Thêm BatchNorm để hội tụ nhanh
                bottom_layers.append(nn.ReLU())
                in_dim = dim
            self.bottom_mlp = nn.Sequential(*bottom_layers)
            assert bottom_mlp_dims[-1] == embed_dim, "Output của Bottom MLP phải bằng với embed_dim"
        
        # top mlp
        total_interacting_features = self.num_cat_features + 1
        
        # Tính số lượng cặp tương tác (n * (n-1) / 2) -> 27 * 26 / 2 = 351
        num_interactions = (total_interacting_features * (total_interacting_features - 1)) // 2
        
        # Đầu vào Top MLP = vector dense_out (64) + các cặp tương tác (351) = 415
        interaction_dim = embed_dim + num_interactions
        
        top_layers = []
        in_dim = interaction_dim
        for dim in top_mlp_dims:
            top_layers.append(nn.Linear(in_dim, dim))
            top_layers.append(nn.BatchNorm1d(dim))
            top_layers.append(nn.ReLU())
            top_layers.append(nn.Dropout(0.2))
            in_dim = dim
        top_layers.append(nn.Linear(in_dim, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, grouped_cat_x):
        # --- 1. Đi qua Bottom MLP ---
        dense_out = self.bottom_mlp(dense_x) # Kích thước: [Batch, 64]
        
        # --- 2. Đi qua TAME Embedding ---
        cat_emb = self.tame_emb(grouped_cat_x) # Kích thước: [Batch, 26, 64]
        
        # --- 3. Chuẩn bị ma trận Tương tác ---
        # Thêm 1 chiều vào dense_out -> [Batch, 1, 64]
        dense_out_unsqueeze = dense_out.unsqueeze(1) 
        
        # Ghép nối dọc theo chiều cột (dim=1) -> stacked_emb kích thước: [Batch, 27, 64]
        stacked_emb = torch.cat([cat_emb, dense_out_unsqueeze], dim=1) 
        
        # --- 4. Tính Dot Product Interactions (BMM) ---
        interaction_matrix = torch.bmm(stacked_emb, stacked_emb.transpose(1, 2)) # [Batch, 27, 27]
        
        # Lấy phần tam giác trên (không tính đường chéo)
        num_features = stacked_emb.size(1)
        row_idx, col_idx = torch.triu_indices(num_features, num_features, offset=1)
        interactions = interaction_matrix[:, row_idx, col_idx] # Kích thước: [Batch, 351]
        
        # --- 5. Nối dense_out gốc và interactions ---
        concat_x = torch.cat([dense_out, interactions], dim=1) # [Batch, 64 + 351 = 415]
        
        # --- 6. Đi qua Top MLP ---
        out = self.top_mlp(concat_x)
        
        return out.squeeze(1)

In [6]:
import json

# Đọc lại vocab_sizes để nạp vào mô hình
with open("/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_vocab_sizes.json", "r") as f:
    vocab_sizes = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

model = TAME_DLRM(
    vocab_sizes=vocab_sizes,
    num_dense_features=len(dense_cols),
    embed_dim=64,
    # Bottom MLP xử lý 13 biến Dense. Lớp cuối CẦN chốt ở 64 chiều.
    bottom_mlp_dims=[256, 128, 64], 
    # Top MLP xử lý 415 chiều tương tác. Bạn có thể tự do tăng giảm kích thước ở đây.
    top_mlp_dims=[256, 128, 64]     
)

if num_gpus > 1:
    model = nn.DataParallel(model)

model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)


Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [7]:
import torch
from torch.utils.flop_counter import FlopCounterMode

print(f"{'Tên Component / Lớp (Layer)':<45} | {'Số lượng Params':<15}")
print("-" * 65)

total_params = 0

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    params = parameter.numel()
    clean_name = name.replace(".weight", "").replace(".bias", " (bias)")
    print(f"{clean_name:<45} | {params:,}")
    total_params += params

print("-" * 65)
print(f"{'TỔNG CỘNG THAM SỐ HUẤN LUYỆN:':<45} | {total_params:,}")
print("="*65)


print("Đang phân tích số lượng phép tính (FLOPs)...")
model.eval()

dummy_dense = torch.randn(1, len(dense_cols)).to(device)
dummy_cat = {
    dim: torch.zeros(1, len(cat_cols), dtype=torch.long).to(device) 
    for dim in ["64", "32", "16", "8", "4", "2"]
}

flop_counter = FlopCounterMode(model, display=True)
with flop_counter:
    _ = model(dummy_dense, dummy_cat)

Tên Component / Lớp (Layer)                   | Số lượng Params
-----------------------------------------------------------------
tame_emb.embedding_blocks.64                  | 554,179,776
tame_emb.embedding_blocks.32                  | 259,585,472
tame_emb.embedding_blocks.16                  | 131,691,008
tame_emb.embedding_blocks.8                   | 43,897,000
tame_emb.embedding_blocks.4                   | 8,779,404
tame_emb.embedding_blocks.2                   | 2,157,628
bottom_mlp.0                                  | 3,328
bottom_mlp.0 (bias)                           | 256
bottom_mlp.1                                  | 256
bottom_mlp.1 (bias)                           | 256
bottom_mlp.3                                  | 32,768
bottom_mlp.3 (bias)                           | 128
bottom_mlp.4                                  | 128
bottom_mlp.4 (bias)                           | 128
bottom_mlp.6                                  | 8,192
bottom_mlp.6 (bias)                     

/tmp/ipykernel_163/1367594755.py:31: UserWarning: mods argument is not needed anymore, you can stop passing it
  flop_counter = FlopCounterMode(model, display=True)


Module                     FLOP    % Total
---------------------  --------  ---------
TAME_DLRM              476.416K    100.00%
 - aten.addmm          383.104K     80.41%
 - aten.bmm             93.312K     19.59%
 TAME_DLRM.bottom_mlp   88.576K     18.59%
  - aten.addmm          88.576K     18.59%
 TAME_DLRM.top_mlp     294.528K     61.82%
  - aten.addmm         294.528K     61.82%


In [7]:
print("Bắt đầu huấn luyện TAME_DLRM...")
best_val_auc = 0.0 
save_path = "best_tame_dlrm_model.pth"
EPOCHS = 5

torch.cuda.reset_peak_memory_stats()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    total_ram_gb = 0.0
    total_vram_gb = 0.0
    batch_count = 0
    
    train_targets, train_preds = [], []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")
    for dense_x, grouped_cat_x, labels in train_bar:
        dense_x = dense_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        
        # Push dict of tensors to GPU
        grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

        optimizer.zero_grad()
        logits = model(dense_x, grouped_cat_x)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())

        current_ram = psutil.virtual_memory().used / (1024**3)
        current_vram = torch.cuda.memory_allocated() / (1024**3)
        
        total_ram_gb += current_ram
        total_vram_gb += current_vram
        batch_count += 1
        
        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # val
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, grouped_cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)
            grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

            logits = model(dense_x, grouped_cat_x)
            loss = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)
    
    avg_ram = total_ram_gb / batch_count
    avg_vram = total_vram_gb / batch_count
    max_vram = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"Tài nguyên Epoch {epoch+1}:")
    print(f"- RAM trung bình:  {avg_ram:.2f} GB")
    print(f"- VRAM trung bình: {avg_vram:.2f} GB")
    print(f"- VRAM đỉnh điểm:  {max_vram:.2f} GB")
    
    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

    if val_auc > best_val_auc:
        print(f"Best Val AUC: {val_auc:.4f}. Đang lưu model...")
        best_val_auc = val_auc
        torch.save(model.state_dict(), save_path)
    else:
        print(f"Val AUC không cải thiện.")
    gc.collect()

Bắt đầu huấn luyện TAME_DLRM...


[TRAIN] Epoch 1/5:   0%|          | 0/4476 [00:02<?, ?it/s]

[VAL] Epoch 1/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 1:
- RAM trung bình:  63.60 GB
- VRAM trung bình: 14.94 GB
- VRAM đỉnh điểm:  18.67 GB
Epoch 1: 
Train Loss: 0.3898 | Train AUC: 0.8533 || Val Loss: 0.3732 | Val AUC: 0.8658

Best Val AUC: 0.8658. Đang lưu model...


[TRAIN] Epoch 2/5:   0%|          | 0/4476 [00:01<?, ?it/s]

...

[VAL] Epoch 4/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 4:
- RAM trung bình:  87.14 GB
- VRAM trung bình: 14.94 GB
- VRAM đỉnh điểm:  18.67 GB
Epoch 4: 
Train Loss: 0.3279 | Train AUC: 0.8992 || Val Loss: 0.4022 | Val AUC: 0.8484

Val AUC không cải thiện.


[TRAIN] Epoch 5/5:   0%|          | 0/4476 [00:01<?, ?it/s]

[VAL] Epoch 5/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 5:
- RAM trung bình:  87.25 GB
- VRAM trung bình: 14.94 GB
- VRAM đỉnh điểm:  18.67 GB
Epoch 5: 
Train Loss: 0.3150 | Train AUC: 0.9071 || Val Loss: 0.4281 | Val AUC: 0.8303

Val AUC không cải thiện.


In [8]:
import torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

test_file ="/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/test.parquet"
test_hf = load_dataset("parquet", data_files=test_file, split="train")
test_dataset = TAMECriteoDataset(test_hf, dense_cols, cat_cols, "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_ivs_dictionary.json")

test_loader = DataLoader(
    test_dataset, 
    batch_size=8192, 
    shuffle=False, 
    num_workers=8, 
    prefetch_factor=2,
    persistent_workers=True, 
    pin_memory=True
)


print("Bắt đầu đánh giá trên tập TEST và đo Latency...")

save_path = "best_tame_dlrm_model.pth"
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

total_latency_ms = 0.0
total_samples = 0
latency_batch_count = 0
WARMUP_BATCHES = 5

test_loss = 0.0
test_targets, test_preds = [], []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="[TEST] Evaluating")
    for i, (dense_x, grouped_cat_x, labels) in enumerate(test_bar):
        # Đẩy dữ liệu lên GPU
        dense_x = dense_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

        batch_size = dense_x.size(0)

        # --- ĐO THỜI GIAN CHỈ DÀNH RIÊNG CHO FORWARD PASS ---
        start_event.record()
        logits = model(dense_x, grouped_cat_x)
        end_event.record()
        
        # --- TÍNH TOÁN METRIC (Bên ngoài vùng bấm giờ) ---
        loss = criterion(logits, labels)
        test_loss += loss.item()

        probs = torch.sigmoid(logits)
        test_targets.extend(labels.cpu().numpy())
        test_preds.extend(probs.cpu().numpy())
        
        # Đồng bộ GPU để chốt thời gian chính xác
        torch.cuda.synchronize()
        
        # Chỉ cộng dồn thời gian sau khi GPU đã Warm-up xong
        if i >= WARMUP_BATCHES:
            batch_latency = start_event.elapsed_time(end_event)
            total_latency_ms += batch_latency
            total_samples += batch_size
            latency_batch_count += 1

# 2. Tính toán Metric cuối cùng
avg_test_loss = test_loss / len(test_loader)
test_auc = roc_auc_score(test_targets, test_preds)

# Tính Latency trung bình
if latency_batch_count > 0:
    avg_batch_latency = total_latency_ms / latency_batch_count
    avg_inference_per_sample = total_latency_ms / total_samples
else:
    avg_batch_latency = 0
    avg_inference_per_sample = 0

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test AUC:  {test_auc:.4f}")

print(f"Latency 1 Batch (Size {test_loader.batch_size}): {avg_batch_latency:.2f} ms")
print(f"Latency 1 Dòng (Per Sample):      {avg_inference_per_sample:.4f} ms")

Generating train split: 0 examples [00:00, ? examples/s]

Đang tải TAME Dictionary vào bộ nhớ...
Bắt đầu đánh giá trên tập TEST và đo Latency...


[TEST] Evaluating:   0%|          | 0/561 [00:01<?, ?it/s]

Test Loss: 0.3719
Test AUC:  0.8672
Latency 1 Batch (Size 8192): 1.90 ms
Latency 1 Dòng (Per Sample):      0.0002 ms
